# V5 CatBoost capacity and precision challenger

Retrospective synthetic-data experiment, not a promoted fraud detector. Reuses V4 causal features and excludes all 2026 final-test rows. Four bounded fits compare positive-class weights and missingness flags.

Fit uses all positives plus a deterministic 10% negative sample with inverse inclusion weights. Selection, threshold selection, and exploratory validation retain natural prevalence and all rows. Median imputation sees only the sampled fit period. Native categorical statistics use chronological input.

The source label maturity, timestamp meaning, and field availability before authorization are unverified. No source/scoring table, schema, UC function, registry model, or endpoint is created. MLflow stores aggregate metrics only; source identifiers never enter the predictor matrix or reports.


In [ ]:
%run ./train_v4

In [ ]:
from datetime import datetime, timedelta
from pyspark.sql import types as T
stamp = datetime(2024, 1, 2, 13)
columns = "transaction_id string, customer_id string, transaction_date timestamp, is_fraud boolean, amount double, currency string, amount_usd double, transaction_type string, channel string, transaction_country string, merchant_category string, transaction_category string, merchant_name string, latitude double, longitude double"
fixture_rows = [(name, "c", date, False, 50.0, "USD", None, "Payment", "App", "MX", None, "Purchase", "M", 19.4, -99.1) for name, date in [("before",stamp-timedelta(hours=1)),("same_a",stamp),("same_b",stamp),("sealed",datetime(2026,1,1))]]
fixture_frame = spark.createDataFrame(fixture_rows, columns)
fixture_result = build_feature_frame(spark, source_frame=fixture_frame).select("transaction_id", "customer_tx_count_1h", "customer_merchant_count_30d").collect()
assert len(fixture_result) == 3
assert all(row["customer_tx_count_1h"] == 1 and row["customer_merchant_count_30d"] == 1 for row in fixture_result if row["transaction_id"].startswith("same"))
print("V4 feature regression fixture passed: same-time exclusion, inclusive lower bound, sealed-test exclusion.")


In [ ]:
%run ./train_v5

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
smoke_x = pd.DataFrame({"n": np.arange(100, dtype=float), "cat": ["a", "b"] * 50})
smoke_y = np.array([0, 1] * 50)
smoke_model = CatBoostClassifier(iterations=3, depth=2, loss_function="Logloss", eval_metric="PRAUC:type=Classic;use_weights=false", class_weights=[1, 2], has_time=True, allow_writing_files=False, verbose=False, thread_count=2)
smoke_model.fit(smoke_x[:80], smoke_y[:80], cat_features=["cat"], eval_set=(smoke_x[80:],smoke_y[80:]))
assert smoke_model.predict_proba(smoke_x[80:]).shape == (20, 2)
print("CatBoost parameter/runtime smoke passed; toy labels are not performance evidence.")


# Temporal protocol

Fit: before 2025-01-01. Select candidate and stop boosting: January–March 2025. Choose threshold: April–June 2025. Exploratory comparison: July–December 2025, previously inspected. Earlier models trained on the internal periods, so these are not newly independent holdouts. Final test stays sealed.

The notebook measures memory before collecting each feature subset. It has a bounded runtime and early stopping. Scores are not calibrated probabilities. Thresholds respect complete score ties; 20%, 50%, and 80% precision targets require adequate alert support and recall. No feasible threshold is a valid negative result.


## Recorded execution — 2026-10-04

Run 457664911930657 completed successfully. C3 (positive-class weight 20, missingness/support flags, 33 effective iterations) was selected on January–March. Fit retained all 2,296 fraud labels and sampled negatives for a total of 229,760 rows. Evaluation was unsampled.

July–December exploratory validation: 743,909 rows / 699 fraud labels; average precision 0.00097231; ROC-AUC 0.5038. At the frozen approximately 1% review threshold: 12 fraud labels among 7,468 alerts, precision 0.161%, recall 1.717%. No supported 20%/50%/80% precision target was feasible.

V5 is not promoted. The final test remains sealed. Digital-event SELECT was rechecked and remained blocked. These results provide no useful high-precision improvement; label generation and pre-authorization availability are still unresolved.

See [the aggregate report](../reports/2026-10-04/training_v5_catboost_report.md). Initial smoke configuration was corrected before banking-data fitting; runtime fixtures and the native CatBoost compatibility test passed on the successful run.

In [ ]:
import json
import mlflow
result = run_v5(spark, mlflow, build_feature_frame)
print(json.dumps(result, indent=2, allow_nan=False))
dbutils.notebook.exit(json.dumps(result, allow_nan=False))